# 10.8 Manipulating character strings

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The ability to work with arbitrary sets of character strings is one of the key advantages of AMPL. We describe here AMPL's string concatenation operator and several functions for building up string-valued expressions that can be used anywhere that set members can appear in AMPL expressions, both in model declarations and in expressions evaluated from Python. Further details are provided in [Section A.4.2](../appendix_a/a_4_2_strings_and_regular_expressions.ipynb). For each of them, we also show the corresponding Python operations, which are the natural choice when the strings are handled in a Python script rather than in the model.

We also show how strings built in Python serve purposes other than being set members. This allows a script to, for example, write a different file or set different option values in each pass through a loop, according to information derived from the contents of the loop's collection.

## 10.8.1 String functions and operators

The concatenation operator `&` takes two strings as operands, and returns a string consisting of the left operand followed by the right operand. For example, given the sets `NUTR` and `FOOD` defined by `diet.mod` and the data of [Figure 2-3](../02/2_3_using_the_AMPL_diet_model.ipynb#fig-2-3), you could use concatenation to define a set whose members represent nutrient-food pairs:

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];

param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize Total_Cost:  sum {j in FOOD} cost[j] * Buy[j];

subject to Diet {i in NUTR}:
   n_min[i] <= sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing diet.mod


In [3]:
import pandas as pd

food_df = pd.DataFrame(
    [
        ("BEEF", 3.19, 2, 10),
        ("CHK",  2.59, 2, 10),
        ("FISH", 2.29, 2, 10),
        ("HAM",  2.89, 2, 10),
        ("MCH",  1.89, 2, 10),
        ("MTL",  1.99, 2, 10),
        ("SPG",  1.99, 2, 10),
        ("TUR",  2.49, 2, 10),
    ],
    columns=["FOOD", "cost", "f_min", "f_max"],
).set_index("FOOD")

nutr_df = pd.DataFrame(
    [
        ("A",     700, 20000),
        ("C",     700, 20000),
        ("B1",    700, 20000),
        ("B2",    700, 20000),
        ("NA",      0, 50000),
        ("CAL", 16000, 24000),
    ],
    columns=["NUTR", "n_min", "n_max"],
).set_index("NUTR")

# Nutrient content: one row per nutrient, one column per food
amt_df = pd.DataFrame(
    {
        "BEEF": [60, 20, 10, 15,  938, 295],
        "CHK":  [ 8,  0, 20, 20, 2180, 770],
        "FISH": [ 8, 10, 15, 10,  945, 440],
        "HAM":  [40, 40, 35, 10,  278, 430],
        "MCH":  [15, 35, 15, 15, 1182, 315],
        "MTL":  [70, 30, 15, 15,  896, 400],
        "SPG":  [25, 50, 25, 15, 1329, 370],
        "TUR":  [60, 20, 15, 10, 1397, 450],
    },
    index=["A", "C", "B1", "B2", "NA", "CAL"],
)

In [4]:
ampl.read("diet.mod")
ampl.set_data(food_df, "FOOD")
ampl.set_data(nutr_df, "NUTR")
ampl.param["amt"] = amt_df.stack().to_dict()

ampl.display("NUTR", "FOOD")
ampl.display('setof {i in NUTR, j in FOOD} i & "_" & j')

set NUTR := A C B1 B2 NA CAL;

set FOOD := BEEF CHK FISH HAM MCH MTL SPG TUR;

set  setof {i in NUTR, j in FOOD} i & '_' & j :=
A_BEEF     A_TUR      C_SPG      B1_MTL     B2_MCH     NA_HAM     CAL_FISH
A_CHK      C_BEEF     C_TUR      B1_SPG     B2_MTL     NA_MCH     CAL_HAM
A_FISH     C_CHK      B1_BEEF    B1_TUR     B2_SPG     NA_MTL     CAL_MCH
A_HAM      C_FISH     B1_CHK     B2_BEEF    B2_TUR     NA_SPG     CAL_MTL
A_MCH      C_HAM      B1_FISH    B2_CHK     NA_BEEF    NA_TUR     CAL_SPG
A_MTL      C_MCH      B1_HAM     B2_FISH    NA_CHK     CAL_BEEF   CAL_TUR
A_SPG      C_MTL      B1_MCH     B2_HAM     NA_FISH    CAL_CHK;



This is not a set that you would normally want to define, but it might be useful if you have to read data in which strings like `"B2_BEEF"` appear. In Python, the same strings are built with an f-string:

In [5]:
NUTR = ampl.set["NUTR"].to_list()
FOOD = ampl.set["FOOD"].to_list()

[f"{i}_{j}" for i in NUTR for j in FOOD][:8]

['A_BEEF', 'A_CHK', 'A_FISH', 'A_HAM', 'A_MCH', 'A_MTL', 'A_SPG', 'A_TUR']

Numbers that appear as arguments to `&` are automatically converted to strings. As an example, for a multi-week model you can create a set of generically-named periods `WEEK1`, `WEEK2`, and so forth, by declaring:

```ampl
param T integer > 1;
set WEEKS ordered = setof {t in 1..T} "WEEK" & t;
```

Numeric operands to `&` are always converted to full precision (or equivalently, to `%.0g` format) as defined in [Section A.16](../appendix_a/a_16_printing_and_display_commands.ipynb). The conversion thus produces the expected results for concatenation of numerical constants and of indices that run over sets of integers or constants, as in our example. Full precision conversion of computed fractional values may sometimes yield surprising results, however. The variation `WEEKS_FRAC` below would seem to create a set of members `WEEK0.1`, `WEEK0.2`, and so forth:

In [6]:
%%writefile weeks.mod

param T integer > 1;
set WEEKS ordered = setof {t in 1..T} "WEEK" & t;
set WEEKS_FRAC ordered = setof {t in 1..T} "WEEK" & 0.1*t;
set WEEKS_FIXED ordered = setof {t in 1..T} sprintf("WEEK%3.1f", 0.1*t);

Writing weeks.mod


But the actual set comes out differently:

In [7]:
ampl.read("weeks.mod")
ampl.param["T"] = 4
ampl.display("WEEKS", "WEEKS_FRAC")

set WEEKS := WEEK1 WEEK2 WEEK3 WEEK4;

set WEEKS_FRAC :=
WEEK0.1                   WEEK0.30000000000000004
WEEK0.2                   WEEK0.4;



Because 0.1 cannot be stored exactly in a binary representation, the value of 0.1\*3 is slightly different from 0.3 in "full" precision. There is no easy way to predict this behavior, but it can be prevented by specifying an explicit conversion using `sprintf`. The `sprintf` function does format conversions in the same way as `printf` ([Section A.16](../appendix_a/a_16_printing_and_display_commands.ipynb)), except that the resulting formatted string becomes the function's return value. For our example, `"WEEK" & 0.1*t` is replaced by `sprintf("WEEK%3.1f", 0.1*t)` in the declaration of `WEEKS_FIXED`:

In [8]:
ampl.display("WEEKS_FIXED")

set WEEKS_FIXED := WEEK0.1 WEEK0.2 WEEK0.3 WEEK0.4;



Python's f-strings behave in exactly the same way, and the remedy is the same, an explicit format:

In [9]:
[f"WEEK{0.1 * t}" for t in range(1, 5)], [f"WEEK{0.1 * t:3.1f}" for t in range(1, 5)]

(['WEEK0.1', 'WEEK0.2', 'WEEK0.30000000000000004', 'WEEK0.4'],
 ['WEEK0.1', 'WEEK0.2', 'WEEK0.3', 'WEEK0.4'])

The `length` string function takes a string as argument and returns the number of characters in it. The `match` function takes two string arguments, and returns the first position where the second appears as a substring in the first, or zero if the second never appears as a substring in the first. For example:

In [10]:
ampl.display('{j in FOOD} (length(j), match(j, "H"))')

:    length(j) match(j, 'H')    :=
BEEF      4           0
CHK       3           2
FISH      4           4
HAM       3           1
MCH       3           3
MTL       3           0
SPG       3           0
TUR       3           0
;



The Python counterparts are `len(j)` and `j.find("H")`; positions in Python strings are counted from 0 rather than 1, so that `find()` returns -1, rather than 0, when the substring does not appear:

In [11]:
{j: (len(j), j.find("H") + 1) for j in FOOD}

{'BEEF': (4, 0),
 'CHK': (3, 2),
 'FISH': (4, 4),
 'HAM': (3, 1),
 'MCH': (3, 3),
 'MTL': (3, 0),
 'SPG': (3, 0),
 'TUR': (3, 0)}

The `substr` function takes a string and one or two integers as arguments. It returns a substring of the first argument that begins at the position given by the second argument; it has the length given by the third argument, or extends to the end of the string if no third argument is given. An empty string is returned if the second argument is greater than the length of the first argument, or if the third argument is less than 1. (In Python, `s[start:end]` slices a string in the same way.)

As an example combining several of these functions, suppose that you want to use the model from `diet.mod` but to supply the nutrition amount data in a table like this:

```
NUTR_FOOD   amt_nutr
A_BEEF            60
B1_BEEF           10
CAL_BEEF         295
CAL_CHK          770
...
```

Then instead of the declarations for the sets and the parameter `amt` used in the model,

```ampl
set NUTR;
set FOOD;
param amt {NUTR,FOOD} >= 0;
```

you would declare a set and a parameter to hold the data from the "nonstandard" table:

```ampl
set NUTR_FOOD;
param amt_nutr {NUTR_FOOD} >= 0;
```

To use the model, you need some way to get the data from set `NUTR_FOOD` and parameter `amt_nutr` into sets `NUTR` and `FOOD` and parameter `amt`. One solution is to define the sets, and then the parameter, by AMPL expressions in the model:

In [12]:
%%writefile nutr_food.mod

set NUTR_FOOD;
param amt_nutr {NUTR_FOOD} >= 0;

set NUTR = setof {ij in NUTR_FOOD} substr(ij, 1, match(ij, "_") - 1);
set FOOD = setof {ij in NUTR_FOOD} substr(ij, match(ij, "_") + 1);
param amt {i in NUTR, j in FOOD} = amt_nutr[i & "_" & j];

Writing nutr_food.mod


We build the "nonstandard" table in Python from the data used so far, and assign it to `amt_nutr`:

In [13]:
amt_nutr = {f"{i}_{j}": value for (i, j), value in amt_df.stack().items()}

ampl.reset()
ampl.read("nutr_food.mod")
ampl.set["NUTR_FOOD"] = list(amt_nutr)
ampl.param["amt_nutr"] = amt_nutr

ampl.display("NUTR", "FOOD")
ampl.display('{i in NUTR} amt[i, "BEEF"]')

set NUTR := A C B1 B2 NA CAL;

set FOOD := BEEF CHK FISH HAM MCH MTL SPG TUR;

amt[i,'BEEF'] [*] :=
  A   60
 B1   10
 B2   15
  C   20
CAL  295
 NA  938
;



As an alternative, you can extract the sets and parameter together in Python, and assign them to the original model. The string method `partition("_")` splits each member of `NUTR_FOOD` at the first underscore:

In [14]:
amt = {}
for ij in amt_nutr:
    i, underscore, j = ij.partition("_")
    assert underscore, f"no '_' in {ij}"
    amt[i, j] = amt_nutr[ij]

ampl.reset()
ampl.read("diet.mod")
ampl.set["NUTR"] = list(dict.fromkeys(i for i, j in amt))   # unique, in order
ampl.set["FOOD"] = list(dict.fromkeys(j for i, j in amt))
ampl.param["amt"] = amt

ampl.display('{i in NUTR} amt[i, "BEEF"]')

amt[i,'BEEF'] [*] :=
  A   60
 B1   10
 B2   15
  C   20
CAL  295
 NA  938
;



Under the first alternative, errors such as a missing `"_"` in a member of `NUTR_FOOD` are eventually signaled by AMPL error messages; under the second, the `assert` statement stops the script with an informative message.

AMPL provides two other functions, `sub` and `gsub`, that look for the second argument in the first, like `match`, but that then substitute a third argument for either the first occurrence (`sub`) or all occurrences (`gsub`) found. The second argument of all three of these functions is actually a regular expression; if it contains certain special characters, it is interpreted as a pattern that may match many sub-strings. The pattern `"^B[0-9]+_"`, for example, matches any sub-string consisting of a `B` followed by one or more digits and then an underscore, and occurring at the beginning of a string. Details of these features are given in [Section A.4.2](../appendix_a/a_4_2_strings_and_regular_expressions.ipynb). Here we declare the set of nutrient-food pairs once more, and use `sub` to remove such a prefix, and `gsub` to replace every vowel:

In [15]:
%%writefile nutr_food_set.mod

set NUTR_FOOD = setof {i in NUTR, j in FOOD} i & "_" & j;

Writing nutr_food_set.mod


In [16]:
ampl.read("nutr_food_set.mod")
ampl.display('{ij in NUTR_FOOD: match(ij, "^B[0-9]+_") > 0} (sub(ij, "^B[0-9]+_", ""), gsub(ij, "[AEIOU]", "*"))')

:       sub(ij, '^B[0-9]+_', '') gsub(ij, '[AEIOU]', '*')    :=
B1_BEEF   BEEF                     'B1_B**F'
B1_CHK    CHK                      B1_CHK
B1_FISH   FISH                     'B1_F*SH'
B1_HAM    HAM                      'B1_H*M'
B1_MCH    MCH                      B1_MCH
B1_MTL    MTL                      B1_MTL
B1_SPG    SPG                      B1_SPG
B1_TUR    TUR                      'B1_T*R'
B2_BEEF   BEEF                     'B2_B**F'
B2_CHK    CHK                      B2_CHK
B2_FISH   FISH                     'B2_F*SH'
B2_HAM    HAM                      'B2_H*M'
B2_MCH    MCH                      B2_MCH
B2_MTL    MTL                      B2_MTL
B2_SPG    SPG                      B2_SPG
B2_TUR    TUR                      'B2_T*R'
;



Python's `re` module provides the same regular expressions, with `re.sub()` replacing all occurrences by default, or only the first one when given `count=1`:

In [17]:
import re

{ij: (re.sub(r"^B[0-9]+_", "", ij, count=1), re.sub(r"[AEIOU]", "*", ij))
 for ij in amt_nutr if re.match(r"^B[0-9]+_", ij)}

{'B1_BEEF': ('BEEF', 'B1_B**F'),
 'B1_CHK': ('CHK', 'B1_CHK'),
 'B1_FISH': ('FISH', 'B1_F*SH'),
 'B1_HAM': ('HAM', 'B1_H*M'),
 'B1_MCH': ('MCH', 'B1_MCH'),
 'B1_MTL': ('MTL', 'B1_MTL'),
 'B1_SPG': ('SPG', 'B1_SPG'),
 'B1_TUR': ('TUR', 'B1_T*R'),
 'B2_BEEF': ('BEEF', 'B2_B**F'),
 'B2_CHK': ('CHK', 'B2_CHK'),
 'B2_FISH': ('FISH', 'B2_F*SH'),
 'B2_HAM': ('HAM', 'B2_H*M'),
 'B2_MCH': ('MCH', 'B2_MCH'),
 'B2_MTL': ('MTL', 'B2_MTL'),
 'B2_SPG': ('SPG', 'B2_SPG'),
 'B2_TUR': ('TUR', 'B2_T*R')}

## 10.8.2 String expressions in commands

In AMPL's own command language, string-valued expressions must be enclosed in parentheses to be used in place of literal strings, for example in filenames or in values assigned to options. From Python there is no such distinction: filenames and option values are Python strings, and they can be built with any of Python's string operations.

Here is an example involving filenames. This script uses f-strings to specify files for reading the data in each pass of a loop, and for writing the results. To have some files to read, we first write three versions of the nutrient data, which differ in the upper limit on sodium:

In [18]:
for j, na_max in enumerate([50000, 55000, 60000], start=1):
    nutr_df.assign(n_max=nutr_df["n_max"].where(nutr_df.index != "NA", na_max)).to_csv(f"diet{j}.csv")

In [19]:
ampl.reset()
ampl.read("diet.mod")
ampl.set_data(food_df, "FOOD")
ampl.set["NUTR"] = nutr_df.index.to_list()
ampl.param["amt"] = amt_df.stack().to_dict()

for j in range(1, 4):
    nutr = pd.read_csv(f"diet{j}.csv", index_col=0, keep_default_na=False)   # keep "NA" as a name
    ampl.set_data(nutr)   # new values for n_min and n_max
    ampl.solve(solver="highs")
    assert ampl.solve_result == "solved", ampl.solve_result
    ampl.var["Buy"].to_pandas().to_csv(f"diet{j}.out")

HiGHS 1.15.1: optimal solution; objective 118.0594032
5 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 107.6298637
1 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 101.0133333
2 simplex iterations
0 barrier iterations


The result is to solve `diet.mod` with a series of different data files `diet1.csv`, `diet2.csv`, and `diet3.csv`, and to save the solution to files `diet1.out`, `diet2.out`, and `diet3.out`. The value of the index `j` is converted automatically from a number to a string by the f-string. Here is one of the files written:

In [20]:
print(open("diet3.out").read())

,Buy.val
BEEF,2.0
CHK,10.0
FISH,2.0
HAM,2.0
MCH,10.0
MTL,10.0
SPG,7.33333333333331
TUR,2.0



The following script uses an f-string to specify the value of the option `highs_options`, which contains directions for the HiGHS solver. The loop solves the same problem four times, each using a different pairing of the directives `method=simplex` and `method=ipm`, which choose the algorithm, with `parallel=off` and `parallel=on`; the function `itertools.product()` generates all of the pairings. (These HiGHS options are described on the [HiGHS options page](https://dev.ampl.com/solvers/highs/options.html).)

By default, AMPL passes the solution found by one solve to the solver as the starting point for the next one, so that the later solves would have almost nothing left to do. To compare the algorithms fairly, we first set two AMPL options so that each solve starts from scratch: `send_statuses` set to 0 keeps AMPL from sending the basis statuses of the previous solution, and `reset_initial_guesses` set to 1 sends the original initial values of the variables rather than their current values. Both options are described in [Section 14.2](../14/14_2_retrieving_results_from_solvers.ipynb).

In [21]:
from itertools import product

ampl.option["send_statuses"] = 0
ampl.option["reset_initial_guesses"] = 1

for method, parallel in product(["simplex", "ipm"], ["off", "on"]):
    ampl.solve(solver="highs", highs_options=f"method={method} parallel={parallel}")
    assert ampl.solve_result == "solved", ampl.solve_result

HiGHS 1.15.1:   alg:method = simplex
  alg:parallel = off
HiGHS 1.15.1: optimal solution; objective 101.0133333
2 simplex iterations
0 barrier iterations
HiGHS 1.15.1:   alg:method = simplex
  alg:parallel = on
HiGHS 1.15.1: optimal solution; objective 101.0133333
2 simplex iterations
0 barrier iterations
HiGHS 1.15.1:   alg:method = ipm
  alg:parallel = off
HiGHS 1.15.1: optimal solution; objective 101.0133333
0 simplex iterations
9 barrier iterations
HiGHS 1.15.1:   alg:method = ipm
  alg:parallel = on
HiGHS 1.15.1: optimal solution; objective 101.0133333
0 simplex iterations
9 barrier iterations
